In [ ]:
import json
import os
from pathlib import Path

from fl_prog.utils.constants import DNAME_LATEST
from fl_prog.utils.io import DEFAULT_DPATH_RESULTS

# ===== Select results to plot =====
# change these as needed
TAG = "adni_iid"
dname_results_date = DNAME_LATEST
fpath_adni_merge = Path(os.environ.get("ADNI_MERGE_FILE"))

fpath_json = Path(
    f"{DEFAULT_DPATH_RESULTS}/{dname_results_date}/{TAG}/{TAG}-estimated_params.json"
)
print(fpath_json)
json_content = json.loads(fpath_json.read_text())
col_subject = json_content["settings"]["config"]["cols"]["col_subject"]
cols_biomarker = json_content["settings"]["config"]["cols"]["cols_biomarker"]
col_months = "months"  # scaled back to original months

subjects_by_node = json_content["settings"]["config"]["subjects_by_node"]
time_scaling_factor = json_content["settings"]["config"]["settings"]["config"][
    "max_time"
]
min_max_by_measure = json_content["settings"]["config"]["settings"]["config"][
    "min_max_by_measure"
]
flipped = json_content["settings"]["config"]["settings"]["config"].get("flip", True)
results = json_content["results"]
# results

In [ ]:
from fl_prog.utils.constants import NODE_PREFIX
import pandas as pd


def get_time_shift_by_subject(
    estimated_time_shifts, subjects_by_node, time_scaling_factor=1.0
):

    dfs_time_shifts = []
    for node_id, time_shifts in estimated_time_shifts.items():
        df_time_shifts = pd.DataFrame(
            {
                "participant_id": [
                    str(subject)
                    for subject in subjects_by_node[node_id.removeprefix(NODE_PREFIX)]
                ],
                # "participant_id": subjects_by_node[node_id.removeprefix(NODE_PREFIX)],
                "estimated_time_shift": time_shifts,
            }
        )
        df_time_shifts["estimated_time_shift"] = (
            df_time_shifts["estimated_time_shift"].astype(float) * time_scaling_factor
        )
        dfs_time_shifts.append(df_time_shifts)

    time_shift_by_subject = (
        pd.concat(dfs_time_shifts, ignore_index=True)
        .set_index("participant_id")
        .squeeze()
        .to_dict()
    )
    return time_shift_by_subject


time_shift_by_subject = get_time_shift_by_subject(
    results["federated"]["estimated_time_shifts"],
    subjects_by_node,
    time_scaling_factor=time_scaling_factor,
)
# time_shift_by_subject

# print(min(time_shift_by_subject.values()))
# print(max(time_shift_by_subject.values()))
# print(max(time_shift_by_subject.values()) - min(time_shift_by_subject.values()))

In [ ]:
import numpy as np

data_fitted_models = []

for setup in results:
    for biomarker, k, x0, vertical_shift, scaling_factor in zip(
        cols_biomarker,
        results[setup]["estimated_k_values"],
        results[setup]["estimated_x0_values"],
        results[setup]["estimated_vertical_shifts"],
        results[setup]["estimated_scaling_factors"],
    ):
        months = np.linspace(-0.5, 4, 100)
        values = scaling_factor * (
            1 / (1 + np.exp(-k * (months - x0))) + vertical_shift
        )

        biomarker_min, biomarker_max = min_max_by_measure[biomarker]

        data_fitted_models.extend(
            [
                {
                    "setup": setup,
                    "biomarker": biomarker,
                    "x": month * time_scaling_factor,
                    "y": value,
                    "y_transformed": (
                        value * (biomarker_max - biomarker_min) + biomarker_min
                    ),
                }
                for month, value in zip(months, values)
            ]
        )

df_fitted_models = pd.DataFrame(data_fitted_models)
df_fitted_models

In [ ]:
import numpy as np

estimated_time_shifts_federated = pd.Series(
    get_time_shift_by_subject(
        results["federated"]["estimated_time_shifts"],
        subjects_by_node,
        time_scaling_factor=time_scaling_factor,
    )
).sort_index()

if "centralized" in results:
    estimated_time_shifts_centralized = pd.Series(
        get_time_shift_by_subject(
            results["centralized"]["estimated_time_shifts"],
            subjects_by_node,
            time_scaling_factor=time_scaling_factor,
        )
    ).sort_index()
    # estimated_time_shifts_federated = np.concatenate(
    #     list(results["federated"]["estimated_time_shifts"].values())
    # )
    # estimated_time_shifts_centralized = np.concatenate(
    #     list(results["centralized"]["estimated_time_shifts"].values())
    # )

    print("Correlation between estimated time shifts (federated vs centralized):")
    print(
        np.corrcoef(estimated_time_shifts_federated, estimated_time_shifts_centralized)
    )
else:
    print("Centralized results not available (yet)")

In [ ]:
import seaborn as sns

fig_models = sns.relplot(
    data=df_fitted_models,
    x="x",
    y="y",
    # y="y_transformed",
    hue="biomarker",
    style="setup",
    kind="line",
)

for ax in fig_models.axes.flatten():
    # xticks = ax.get_xticks()
    # ax.set_xticks(xticks)
    # ax.set_xticklabels(xticks * time_scaling_factor)
    ax.set_ylabel("Biomarker value")
    ax.set_xlabel("Months")

In [ ]:
import pandas as pd

if fpath_adni_merge is None or not fpath_adni_merge.exists():
    raise ValueError(
        "ADNI merge file path is not set or does not exist. Please set the ADNI_MERGE_FILE environment variable to the correct path."
    )

data_specific_cols = ["rid", "visit"]

index_cols = ["participant_id_int", "months_scaled"] + data_specific_cols

df_adni = pd.read_csv(
    f"{json_content['settings']['dpath_data']}/{TAG}-merged.tsv",
    sep="\t",
    index_col=index_cols,
    dtype={col: str for col in data_specific_cols},
)

df_adni_long = pd.melt(
    df_adni.reset_index(),
    id_vars=index_cols,
    var_name="feature",
    value_name="value",
)
df_adni_long["months_shifted"] = df_adni_long[["months_scaled", "rid"]].apply(
    lambda row: (
        (row["months_scaled"] * time_scaling_factor) + time_shift_by_subject[row["rid"]]
    ),
    axis="columns",
)

df_demographics = pd.read_csv(
    fpath_adni_merge,
    low_memory=False,
    dtype={"RID": str, "Month": str},
)
df_demographics = df_demographics.query('RID in @df_adni.index.get_level_values("rid")')
df_demographics_baseline = df_demographics.query('VISCODE == "bl"')
df_adni_long["group"] = df_adni_long["rid"].map(
    df_demographics_baseline.set_index("RID")["DX_bl"].to_dict()
)

df_adni_long

In [ ]:
import seaborn as sns

sns.set_context("notebook")

fig_model_fits = sns.relplot(
    data=df_fitted_models,
    x="x",
    y="y",
    style="setup",
    col="biomarker",
    col_wrap=2,
    kind="line",
    aspect=2,
)

for i_ax, (biomarker, ax) in enumerate(fig_model_fits.axes_dict.items()):
    df_adni_long_subset = df_adni_long.query(f"feature == '{biomarker}'")
    # palette = {
    #     participant_id: sns.color_palette()[i_ax]
    #     for participant_id in df_adni_long_subset["participant_id_int"].unique()
    # }
    hue_order = ["CN", "SMC", "EMCI", "LMCI", "AD"]
    palette = {group: f"C{hue_order.index(group)}" for group in hue_order}
    sns.lineplot(
        data=df_adni_long_subset,
        x="months_shifted",
        y="value",
        hue="group",
        palette=palette,
        hue_order=hue_order,
        style="participant_id_int",
        alpha=0.2,
        ax=ax,
        legend=False,
    )

    ax.set_ylabel("Biomarker value")
    ax.set_xlabel("Months")

    # xticks = ax.get_xticks()
    # ax.set_xticks(xticks)
    # ax.set_xticklabels(xticks * time_scaling_factor)

In [ ]:
import numpy as np

hue_order = ["CN", "SMC", "EMCI", "LMCI", "AD"]

df_time_shifts = estimated_time_shifts_federated.reset_index(
    name="estimated_time_shift"
)

df_time_shifts["group"] = df_time_shifts["index"].map(
    df_demographics_baseline.set_index("RID")["DX_bl"].to_dict()
)
fig_time_shift_kde = sns.displot(
    data=df_time_shifts,
    x="estimated_time_shift",
    hue="group",
    kind="kde",
    hue_order=hue_order,
)

fig_time_shift_box = sns.catplot(
    data=df_time_shifts.sort_values(
        "group", key=lambda x: x.map({group: i for i, group in enumerate(hue_order)})
    ),
    y="estimated_time_shift",
    x="group",
    kind="box",
    hue="group",
    hue_order=hue_order,
)

# ax = fig_time_shift_kde.ax
# xticks = np.asarray(ax.get_xticks())
# ax.set_xticks(xticks)
# ax.set_xticklabels(xticks * time_scaling_factor)

# ax = fig_time_shift_box.ax
# yticks = np.asarray(ax.get_yticks())
# ax.set_yticks(yticks)
# ax.set_yticklabels(yticks * time_scaling_factor)

In [ ]:
data_for_df_params = []
for i_round in results["federated"]["aggregated_params"]:
    round_params = results["federated"]["aggregated_params"][i_round]["params"]
    for i_biomarker in range(len(cols_biomarker)):
        for param_name_prefix in ("x0_values", "parametrizations.k_values.original", "parametrizations.sigma.original"):
            data_for_df_params.append(
                {
                    "round": i_round,
                    "i_biomarker": f"{i_biomarker}",
                    "param_value": round_params[param_name_prefix][i_biomarker],
                    "param_name": param_name_prefix,
                }
            )
    # data_for_df_params.append(
    #     {
    #         "round": i_round,
    #         "i_biomarker": "-1",
    #         "param_value": round_params[],
    #         "param_name": "parametrizations.sigma.original",
    #     }
    # )

df_params = pd.DataFrame(data_for_df_params)
# df_params

In [ ]:
import seaborn as sns

sns.relplot(
    data=df_params,
    x="round",
    y="param_value",
    hue="i_biomarker",
    row="param_name",
    kind="line",
    facet_kws={"sharey": False},
)